<a href="https://colab.research.google.com/github/mariaaapetrovskaya/complingua/blob/main/gesture_classification_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install av torchcodec open_clip_torch

import open_clip
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset

dataset = load_dataset("mapetrovska/gesturedataset")
sample = dataset['train'][0]
video_data = sample['video']
true_label = sample['label']


frames = []
for frame_tensor in video_data:
    arr = frame_tensor.permute(1, 2, 0).cpu().numpy()
    frame = Image.fromarray(arr)
    frames.append(frame)

print(f"Всего кадров: {len(frames)}")
if len(frames) == 0:
    raise RuntimeError("Не удалось извлечь")

# средний кадр
middle_frame = frames[len(frames)//2]

model_name = 'EVA02-B-16'
pretrained = 'merged2b_s8b_b131k'

try:
    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    tokenizer = open_clip.get_tokenizer(model_name)
    print(f"Модель {model_name} с весами {pretrained} загружена.")
except RuntimeError as e:
    print(f"Неудалось загрузить {model_name} с тегом {pretrained}. Ошибка: {e}")
    print( "ViT-B-32")
    model_name = 'ViT-B-32'
    pretrained = 'laion2b_e16'
    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    tokenizer = open_clip.get_tokenizer(model_name)

model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Модель {model_name} {device}")


image = preprocess(middle_frame).unsqueeze(0).to(device)

text = tokenizer([str(i) for i in range(8)]).to(device)


with torch.no_grad(), torch.cuda.amp.autocast():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

pred_label = probs[0].argmax().item()
print(f"Средний кадр (индекс {len(frames)//2})")
print(f"Истина: {true_label}, Предсказание: {pred_label}")

Resolving data files:   0%|          | 0/101 [00:00<?, ?it/s]

Всего кадров: 18


open_clip_model.safetensors:   0%|          | 0.00/299M [00:00<?, ?B/s]

Модель EVA02-B-16 с весами merged2b_s8b_b131k загружена.
Модель EVA02-B-16 загружена на cuda


/tmp/ipykernel_19484/1194028072.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


Средний кадр (индекс 9)
Истина: 0, Предсказание: 5


In [ ]:

!pip install av torchcodec open_clip_torch tqdm

import open_clip
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm


dataset = load_dataset("mapetrovska/gesturedataset")


model_name = 'EVA02-B-16'
pretrained = 'merged2b_s8b_b131k'
model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
tokenizer = open_clip.get_tokenizer(model_name)
model.eval()


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Модель {model_name} загружена на {device}")


text_labels = [str(i) for i in range(8)]
text = tokenizer(text_labels).to(device)

with torch.no_grad():
    text_features = model.encode_text(text)
    text_features /= text_features.norm(dim=-1, keepdim=True)

correct = 0
total = 0

for sample in tqdm(dataset['train'], desc="Обработка видео"):
    video_data = sample['video']
    true_label = sample['label']


    frames = []
    try:
        for frame_tensor in video_data:

            arr = frame_tensor.permute(1, 2, 0).cpu().numpy()
            frame = Image.fromarray(arr)
            frames.append(frame)
    except Exception as e:
        print(f"Ошибка при извлечении кадров {e}")
        continue

    if len(frames) == 0:
        continue


    middle_frame = frames[len(frames) // 2]

    image = preprocess(middle_frame).unsqueeze(0).to(device)


    with torch.no_grad(), torch.cuda.amp.autocast():
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

    pred_label = probs[0].argmax().item()

    if pred_label == true_label:
        correct += 1
    total += 1


    del frames, middle_frame, image


print(f"\nВсего обработано видео: {total}")
print(f"Правильных предсказаний: {correct}")
if total > 0:
    print(f"Accuracy: {correct/total:.4f}")
else:
    print("Не удалось")

Resolving data files:   0%|          | 0/101 [00:00<?, ?it/s]

Модель EVA02-B-16 загружена на cuda


Обработка видео:   0%|          | 0/101 [00:00<?, ?it/s]/tmp/ipykernel_19484/3381508391.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():
Обработка видео: 100%|██████████| 101/101 [01:31<00:00,  1.10it/s]


Всего обработано видео: 101
Правильных предсказаний: 9
Accuracy (один центральный кадр): 0.0891
